In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path(r"D:\studia\DermaAI\Backend\DermaAPI\derma.db")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
if not DB_PATH.exists():
    display(Markdown(f"**Błąd:** Nie znaleziono pliku bazy danych pod ścieżką: `{DB_PATH}`"))
else:
    display(Markdown(f"**Sukces:** Odnaleziono bazę danych: `{DB_PATH}`"))
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    
    display(Markdown(f"### Znalezione tabele w bazie: **{tables}**"))

In [ ]:
if DB_PATH.exists() and 'conn' in locals():
    for table_name in tables:
        display(Markdown(f"---"))
        display(Markdown(f"## Tabela: `{table_name}`"))
        
        # 1. Pobranie i wyświetlenie schematu tabeli (Typy danych, klucze itp.)
        display(Markdown(f"### Schemat tabeli:"))
        try:
            schema_df = pd.read_sql(f'PRAGMA table_info("{table_name}")', conn)
            # Wybieramy najistotniejsze kolumny ze schematu:
            # cid (id kolumny), name (nazwa), type (typ np. TEXT, INTEGER), 
            # notnull (czy wymagane: 1=tak, 0=nie), dflt_value (domyślna wartość), pk (czy klucz główny)
            display(schema_df[['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk']])
        except Exception as e:
            display(Markdown(f"Błąd pobierania schematu dla `{table_name}`: `{e}`"))
            
        # 2. Pobranie i wyświetlenie danych z tabeli
        display(Markdown(f"### Zawartość (Wartości w tabeli):"))
        try:
            query = f'SELECT * FROM "{table_name}"'
            df = pd.read_sql(query, conn)
            
            if df.empty:
                display(Markdown(f"*Tabela `{table_name}` jest pusta (brak rekordów).*"))
            else:
                display(Markdown(f"Liczba rekordów: **{len(df)}**"))
                display(df)
                
        except Exception as e:
            display(Markdown(f"Błąd podczas odczytu danych z tabeli `{table_name}`: `{e}`"))
            
    # Zamknięcie połączenia po przetworzeniu wszystkich tabel
    conn.close()
    display(Markdown("---"))
    display(Markdown("**Analiza bazy zakończona. Połączenie zostało zamknięte.**"))

In [ ]:
import sqlite3
from pathlib import Path
import uuid
from datetime import datetime

In [ ]:
import sqlite3
from pathlib import Path
import uuid
from datetime import datetime
from loguru import logger

# Path to the SQLite database in the .NET backend
DB_PATH = Path(r"D:\studia\DermaAI\Backend\DermaAPI\derma.db")

# Dictionary of available tags (matching the C# enum)
AVAILABLE_TAGS = {
    0: "PoczatekCyklu",
    1: "SloneJedzenie",
    2: "OstreJedzenie",
    3: "FastFood",
    4: "Alkohol"
}

def print_available_tags():
    """Displays a clean list of available tags."""
    logger.info("Displaying available system tags:")
    print("Available tags:")
    for key, name in AVAILABLE_TAGS.items():
        print(f"   [{key}] -> '{name}'")

def insert_new_skin_analysis_with_tags(user_id, image_url, notes, tags_to_add=None):
    """
    Inserts a new record into the SkinAnalysis table and multiple tags into SkinAnalysisTag.
    
    The tags_to_add parameter can be:
    - A list of names or indices, e.g., ["FastFood", "Alkohol"] or [3, 4]
    - A dictionary of pairs {tag: value}, e.g., {"FastFood": 1, "Alkohol": 1}
    """
    logger.info("Initializing process to insert a new skin analysis record...")
    
    if tags_to_add is None:
        tags_to_add = []

    # Reversed dictionary for quick index lookup by tag name
    name_to_id = {v.lower(): k for k, v in AVAILABLE_TAGS.items()}
    processed_tags = []
    
    # Process user input into a universal format (index, value)
    if isinstance(tags_to_add, dict):
        for t_key, t_val in tags_to_add.items():
            idx = name_to_id.get(t_key.lower()) if isinstance(t_key, str) else t_key
            if idx is not None and idx in AVAILABLE_TAGS:
                processed_tags.append((idx, t_val))
            else:
                logger.warning(f"Ignoring unknown tag key: '{t_key}'")
    elif isinstance(tags_to_add, list):
        for item in tags_to_add:
            # If item is a tuple/list like [name, value], e.g., ["FastFood", 1]
            if isinstance(item, (list, tuple)) and len(item) == 2:
                t_key, t_val = item
                idx = name_to_id.get(t_key.lower()) if isinstance(t_key, str) else t_key
            else:
                # If only name or index is provided, default value is set to 1
                idx = name_to_id.get(item.lower()) if isinstance(item, str) else item
                t_val = 1
                
            if idx is not None and idx in AVAILABLE_TAGS:
                processed_tags.append((idx, t_val))
            else:
                logger.warning(f"Ignoring unknown tag item: '{item}'")

    # 1. Check database availability
    if not DB_PATH.exists():
        logger.error(f"Database file not found at path: {DB_PATH}")
        return None

    logger.success(f"Database found at: {DB_PATH}")

    try:
        # 2. Establish connection
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        logger.debug("Successfully connected to the SQLite database.")
        
        analysis_id = str(uuid.uuid4()).upper()
        upload_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:23]
        status = 0  # 0 = Pending / New
        
        # Log main details before insertion
        logger.info(f"Preparing to insert SkinAnalysis -> ID: {analysis_id}, UserId: {user_id}, Status: {status}")
        logger.debug(f"Image URL: {image_url} | Notes: {notes} | Upload Date: {upload_date}")

        # 3. Insert main analysis record
        cursor.execute("""
            INSERT INTO SkinAnalysis (Id, Status, imageUrl, notes, uploadDate, userId)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (analysis_id, status, image_url, notes, upload_date, user_id))
        
        logger.success("Main SkinAnalysis record inserted successfully.")

        # 4. Insert all processed tags in a loop
        for tag_index, tag_value in processed_tags:
            tag_id = str(uuid.uuid4()).upper()
            tag_name = AVAILABLE_TAGS[tag_index]
            
            cursor.execute("""
                INSERT INTO SkinAnalysisTag (Id, SkinAnalysisId, Tag, Value)
                VALUES (?, ?, ?, ?)
            """, (tag_id, analysis_id, tag_index, tag_value))
            
            logger.info(f"Inserted Tag -> TagID: {tag_id}, Name: {tag_name} (Index: {tag_index}), Value: {tag_value}")
        
        conn.commit()
        conn.close()
        
        logger.success(f"Transaction committed. Successfully added new analysis ID: {analysis_id} with {len(processed_tags)} tag(s).")
        return analysis_id
        
    except Exception as e:
        logger.exception(f"An error occurred while writing to the database: {e}")
        if 'conn' in locals() and conn:
            conn.rollback()
            conn.close()
        return None

sample_user_id = "5A281AF7-CDAA-4F32-9299-8A40F4EF0985"
sample_image = r"D:\studia\DermaAI\Models\yolo\images\raw\test-image-multi-tags.png"
sample_notes = "Test with loguru logger and English comments"

# Display available tags
print_available_tags()
print("-" * 40)

# Call function with a list of tags
new_id = insert_new_skin_analysis_with_tags(
    user_id=sample_user_id,
    image_url=sample_image,
    notes=sample_notes,
    tags_to_add=["FastFood", "Alkohol", "OstreJedzenie"]
)

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from loguru import logger

# Path to the SQLite database in the .NET backend
DB_PATH = Path(r"D:\studia\DermaAI\Backend\DermaAPI\derma.db")

# Dictionary of available tags (matching the C# enum)
AVAILABLE_TAGS = {
    0: "PoczatekCyklu",
    1: "SloneJedzenie",
    2: "OstreJedzenie",
    3: "FastFood",
    4: "Alkohol"
}

def display_single_analysis_details(conn, analysis_id):
    """Helper function that retrieves and displays complete information for a specific analysis ID."""
    
    # 1. SkinAnalysis
    df_analysis = pd.read_sql('SELECT * FROM SkinAnalysis WHERE Id = ?', conn, params=(analysis_id,))
    display(Markdown(f"#### 1. SkinAnalysis (ID: `{analysis_id}`)"))
    if df_analysis.empty:
        display(Markdown("*No record found.*"))
        return
    display(df_analysis)
        
    # 2. AnalysisResult
    df_result = pd.read_sql('SELECT * FROM AnalysisResult WHERE skinAnalysisId = ?', conn, params=(analysis_id,))
    display(Markdown("#### 2. AnalysisResult"))
    if df_result.empty:
        display(Markdown("*No AI results found.*"))
    else:
        display(df_result)
        result_ids = df_result['Id'].tolist()
        
        # 3. Detection
        placeholders = ','.join(['?'] * len(result_ids))
        df_detections = pd.read_sql(f'SELECT * FROM Detection WHERE AnalysisResultId IN ({placeholders})', conn, params=result_ids)
        display(Markdown(f"#### 3. Detection (Total: {len(df_detections)})"))
        if not df_detections.empty:
            display(df_detections)
        else:
            display(Markdown("*No detections found.*"))

    # 4. SkinAnalysisTag
    df_tags = pd.read_sql('SELECT * FROM SkinAnalysisTag WHERE SkinAnalysisId = ?', conn, params=(analysis_id,))
    display(Markdown("#### 4. SkinAnalysisTag"))
    if df_tags.empty:
        display(Markdown("*No tags found.*"))
    else:
        df_tags['TagName'] = df_tags['Tag'].map(AVAILABLE_TAGS).fillna("Unknown")
        display(df_tags[['Id', 'SkinAnalysisId', 'Tag', 'TagName', 'Value']])

def get_analysis_details_by_id(target_id):
    """Retrieves and displays complete information by identifying whether target_id is Analysis ID, AnalysisResult ID, or User ID."""
    logger.info(f"Starting search process for target ID: {target_id}")

    if not DB_PATH.exists():
        display(Markdown("**Error:** Database file not found."))
        return

    try:
        conn = sqlite3.connect(DB_PATH)
        display(Markdown("---"))
        
        cursor = conn.cursor()
        
        analysis_ids_to_process = []
        id_type_description = ""

        # 1. Check if target_id is a direct SkinAnalysis ID
        cursor.execute('SELECT COUNT(*) FROM SkinAnalysis WHERE Id = ?', (target_id,))
        is_analysis_id = cursor.fetchone()[0] > 0

        if is_analysis_id:
            id_type_description = "SkinAnalysis ID (Direct Analysis identifier)"
            logger.info(f"Target ID '{target_id}' recognized as a {id_type_description}.")
            analysis_ids_to_process = [target_id]
        else:
            # 2. Check if target_id is an AnalysisResult ID
            cursor.execute('SELECT skinAnalysisId FROM AnalysisResult WHERE Id = ?', (target_id,))
            res = cursor.fetchone()
            if res:
                id_type_description = "AnalysisResult ID (AI Result identifier)"
                logger.info(f"Target ID '{target_id}' recognized as an {id_type_description}.")
                analysis_ids_to_process = [res[0]]
            else:
                # 3. Check if target_id is a User ID
                cursor.execute('SELECT Id FROM SkinAnalysis WHERE userId = ?', (target_id,))
                user_analyses = [row[0] for row in cursor.fetchall()]
                if user_analyses:
                    id_type_description = f"User ID (Owner of {len(user_analyses)} analysis/analyses)"
                    logger.info(f"Target ID '{target_id}' recognized as a {id_type_description}.")
                    analysis_ids_to_process = user_analyses

        # Display very clear info about the ID type and results
        display(Markdown(f"### ID Type Recognized: **{id_type_description}**"))
        display(Markdown(f"Target ID evaluated: `{target_id}`"))

        if analysis_ids_to_process:
            for idx, analysis_id in enumerate(analysis_ids_to_process, 1):
                display(Markdown("---"))
                if len(analysis_ids_to_process) > 1:
                    display(Markdown(f"### Analysis #{idx} of {len(analysis_ids_to_process)}"))
                display_single_analysis_details(conn, analysis_id)
        else:
            logger.warning(f"No records found for ID '{target_id}' in any table.")
            display(Markdown(f"**Warning:** No records found for ID `{target_id}` as Analysis ID, AnalysisResult ID, or User ID."))

        conn.close()
        logger.success(f"Operation completed successfully for ID: {target_id}")
        display(Markdown("---"))

    except Exception as e:
        logger.exception(f"An error occurred: {e}")
        display(Markdown(f"**Error:** `{e}`"))
        if 'conn' in locals() and conn:
            conn.close()

TARGET_ID = "8F7478CA-390A-41C7-BD50-34CC44119B77" 
get_analysis_details_by_id(TARGET_ID)

In [ ]:
import sqlite3
from pathlib import Path
from IPython.display import display, Markdown
from loguru import logger

# Path to the SQLite database in the .NET backend
DB_PATH = Path(r"D:\studia\DermaAI\Backend\DermaAPI\derma.db")

def preview_and_delete_by_id(target_id, confirm=False):
    """
    Previews and deletes records associated with the given ID (either Analysis ID or User ID).
    If confirm=False, it only displays information and logs what would be deleted.
    If confirm=True, it executes the cascading deletion.
    """
    logger.info(f"Initiating deletion check for target ID: {target_id} (Confirmation mode: {confirm})")

    if not DB_PATH.exists():
        logger.error(f"Database file not found at path: {DB_PATH}")
        display(Markdown("**Error:** Database file not found."))
        return

    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()

        # 1. Check if target_id is an Analysis ID (SkinAnalysis.Id)
        cursor.execute('SELECT Id, userId, imageUrl FROM SkinAnalysis WHERE Id = ?', (target_id,))
        analysis_record = cursor.fetchone()

        if analysis_record:
            analysis_id, user_id, image_url = analysis_record
            logger.info(f"Target ID '{target_id}' identified as an ANALYSIS ID.")
            
            # Find related records
            cursor.execute('SELECT Id FROM AnalysisResult WHERE skinAnalysisId = ?', (analysis_id,))
            result_rows = cursor.fetchall()
            result_ids = [r[0] for r in result_rows]

            # Count related detections
            detection_count = 0
            if result_ids:
                placeholders = ','.join(['?'] * len(result_ids))
                cursor.execute(f'SELECT COUNT(*) FROM Detection WHERE AnalysisResultId IN ({placeholders})', result_ids)
                detection_count = cursor.fetchone()[0]

            # Count related tags
            cursor.execute('SELECT COUNT(*) FROM SkinAnalysisTag WHERE SkinAnalysisId = ?', (analysis_id,))
            tag_count = cursor.fetchone()[0]

            display(Markdown(f"### Deletion Preview: Analysis ID `{analysis_id}`"))
            display(Markdown(f"- **Associated User ID:** `{user_id}`"))
            display(Markdown(f"- **Image URL/Path:** `{image_url}`"))
            display(Markdown(f"- **Related Analysis Results:** `{len(result_ids)}`"))
            display(Markdown(f"- **Related Detections:** `{detection_count}`"))
            display(Markdown(f"- **Related Tags:** `{tag_count}`"))

            if not confirm:
                display(Markdown("---"))
                display(Markdown("**Waiting for confirmation.** To proceed with the deletion, set `confirm=True` and run the cell again."))
                logger.warning(f"Deletion halted for Analysis ID: {analysis_id}. Awaiting user confirmation.")
            else:
                logger.warning(f"CONFIRMED. Executing cascading deletion for Analysis ID: {analysis_id}")
                
                # Cascading delete
                cursor.execute('DELETE FROM SkinAnalysisTag WHERE SkinAnalysisId = ?', (analysis_id,))
                logger.info(f"Deleted {tag_count} tag(s) associated with analysis ID: {analysis_id}")

                if result_ids:
                    cursor.execute(f'DELETE FROM Detection WHERE AnalysisResultId IN ({placeholders})', result_ids)
                    logger.info(f"Deleted {detection_count} detection(s) associated with analysis results.")
                    
                    cursor.execute('DELETE FROM AnalysisResult WHERE skinAnalysisId = ?', (analysis_id,))
                    logger.info(f"Deleted {len(result_ids)} analysis result(s).")

                cursor.execute('DELETE FROM SkinAnalysis WHERE Id = ?', (analysis_id,))
                logger.success(f"Successfully deleted SkinAnalysis record with ID: {analysis_id}")

                conn.commit()
                display(Markdown(f"**Success:** Analysis ID `{analysis_id}` and all its related records have been deleted."))

        else:
            # 2. Check if target_id is a User ID (SkinAnalysis.userId)
            cursor.execute('SELECT Id FROM SkinAnalysis WHERE userId = ?', (target_id,))
            user_analysis_ids = [row[0] for row in cursor.fetchall()]

            if user_analysis_ids:
                logger.info(f"Target ID '{target_id}' identified as a USER ID with {len(user_analysis_ids)} analysis/analyses.")
                
                display(Markdown(f"### Deletion Preview: User ID `{target_id}`"))
                display(Markdown(f"- **Total Analyses to delete:** `{len(user_analysis_ids)}`"))

                if not confirm:
                    display(Markdown("---"))
                    display(Markdown("**Waiting for confirmation.** To proceed with the deletion of this user and ALL their data, set `confirm=True` and run the cell again."))
                    logger.warning(f"Deletion halted for User ID: {target_id}. Awaiting user confirmation.")
                else:
                    logger.warning(f"CONFIRMED. Executing cascading deletion for User ID: {target_id}")

                    for aid in user_analysis_ids:
                        cursor.execute('DELETE FROM SkinAnalysisTag WHERE SkinAnalysisId = ?', (aid,))
                        
                        cursor.execute('SELECT Id FROM AnalysisResult WHERE skinAnalysisId = ?', (aid,))
                        res_rows = cursor.fetchall()
                        r_ids = [r[0] for r in res_rows]
                        if r_ids:
                            ph = ','.join(['?'] * len(r_ids))
                            cursor.execute(f'DELETE FROM Detection WHERE AnalysisResultId IN ({ph})', r_ids)
                            cursor.execute('DELETE FROM AnalysisResult WHERE skinAnalysisId = ?', (aid,))
                        
                        cursor.execute('DELETE FROM SkinAnalysis WHERE Id = ?', (aid,))
                        logger.info(f"Deleted analysis ID: {aid} belonging to user: {target_id}")

                    conn.commit()
                    logger.success(f"Successfully deleted all data for User ID: {target_id}")
                    display(Markdown(f"**Success:** User ID `{target_id}` and all its associated analyses, results, detections, and tags have been deleted."))

            else:
                logger.warning(f"ID '{target_id}' not found in the database as either an Analysis ID or a User ID.")
                display(Markdown(f"**Error:** ID `{target_id}` was not found in the database as an Analysis ID or User ID."))

        conn.close()

    except Exception as e:
        logger.exception(f"An error occurred during deletion process: {e}")
        display(Markdown(f"**Error:** `{e}`"))
        if 'conn' in locals() and conn:
            conn.close()

# Step 1: Run with confirm=False (default) to preview what will be deleted and check logs:
TARGET_ID = "3FA85F64-5717-4562-B3FC-2C963F66AFA6"
preview_and_delete_by_id(TARGET_ID, confirm=False)

# Step 2: Once verified, change confirm=True to execute the deletion:

In [ ]:
preview_and_delete_by_id(TARGET_ID, confirm=True)